# Physical MCU Deployment — Flagship Model on the Silicon Labs BRD2605A (SiWG917)

*An extension added after the original project, once I got hands-on access to a Silicon
Labs BRD2605A (SiWG917) evaluation kit through a hardware workshop during a CFHE
internship.*

The original project compiled a deliberately tiny **4-feature edge model** to
**combinational Verilog** and verified it in RTL simulation. That model is small for a
concrete physical reason: on an FPGA/ASIC the model *is* the circuit, so its size is
bounded by **silicon gate area**.

This notebook adds a **second, different** hardware target: the **full 40-feature
flagship model** compiled to **float C** and run on a physical **Cortex-M4F** at
180 MHz. The point of this notebook is to show *why the same model that is impractical
as combinational logic runs comfortably on the MCU* — and to verify it does so
bit-faithfully, then measure real on-device latency.


## Three hardware targets, three different constraints

| | **FPGA / ASIC** (4-feat edge) | **MCU — SiWG917** (this notebook) | **Server** |
|---|---|---|---|
| Execution | **Combinational** — pure gates, no CPU | **Sequential** — one FPU, many cycles | CPU |
| Memory | none (model = wiring) | ~320 KB SRAM, 8 MB flash | GBs |
| Bounded by | **silicon area** (every compare is gates) | flash bytes (trivial here) | nothing |
| Model | 4 feat, depth 4 | **full 40 feat** | full 40 feat |

**The key idea:** an MCU evaluates the tree ensemble by reusing one comparator
*sequentially* across many clock cycles, so a big model costs *time*, not area — and at
hourly sampling we have billions of spare cycles. An FPGA combinational block pays gate
area for *every* comparison *simultaneously*, so the same big model costs prohibitive
area. That is why the flagship model is trivial on the M4 but the edge path had to shrink
to 4 features. The 4-feature model is not "the weak-hardware model" — it is the only one
that survives synthesis into a small gate-level block.


In [1]:
import numpy as np, pandas as pd, xgboost as xgb, re, subprocess, sys
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score

VITALS = ['HR','O2Sat','Temp','SBP','MAP','DBP','Resp','EtCO2']
FEATURES = (VITALS + [f'{v}_mean3' for v in VITALS] + [f'{v}_mean6' for v in VITALS]
            + [f'{v}_delta3' for v in VITALS] + [f'{v}_missing' for v in VITALS])

# Same feature engineering and seeded patient-level split as 01_sepsis_eda_modeling.
raw = pd.read_csv('data/Dataset.csv')
df = raw.rename(columns={'Patient_ID':'patient_id'}).copy()
g = df.groupby('patient_id', sort=False)
for v in VITALS:
    df[f'{v}_mean3'] = g[v].rolling(3, min_periods=1).mean().reset_index(level=0, drop=True)
    df[f'{v}_mean6'] = g[v].rolling(6, min_periods=1).mean().reset_index(level=0, drop=True)
    df[f'{v}_delta3'] = g[v].diff(3)
    df[f'{v}_missing'] = df[v].isna().astype('int8')
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
_, te = next(gss.split(df, groups=df['patient_id']))
test_df = df.iloc[te]
y_test = test_df['SepsisLabel'].values
print(f'test rows: {len(test_df):,} | features: {len(FEATURES)}')

test rows: 310,997 | features: 40


## The flagship model, and its budget on the board

BRD2605A / SiWG917M: **Cortex-M4F @ 180 MHz**, **~320 KB SRAM**, **8 MB flash**
([datasheet](https://www.silabs.com/documents/public/data-sheets/siwg917-datasheet.pdf)).


In [2]:
bst = xgb.Booster(); bst.load_model('models/sepsis_booster.json')
n_trees = int(bst.attributes()['best_iteration']) + 1   # early-stopping optimum
dtest = xgb.DMatrix(test_df[FEATURES].values, feature_names=FEATURES)
proba = bst.predict(dtest, iteration_range=(0, n_trees))
roc = roc_auc_score(y_test, proba); pr = average_precision_score(y_test, proba)

tdf = bst.trees_to_dataframe(); tdf = tdf[tdf.Tree < n_trees]
n_nodes = len(tdf)
# on-device footprint of the generated C (node struct ~20 B aligned)
flash_kb = n_nodes * 20 / 1024
print(f'Flagship: {n_trees} trees | {len(FEATURES)} features | {n_nodes:,} nodes')
print(f'Test ROC-AUC {roc:.4f} | PR-AUC {pr:.4f}')
print(f'Model in flash ~{flash_kb:.0f} KB of 8192 KB  ({flash_kb/8192:.2%})')
print(f'SRAM at runtime: model is const/flash-resident; only a 40-float input buffer in RAM')

Flagship: 346 trees | 40 features | 20,414 nodes
Test ROC-AUC 0.7256 | PR-AUC 0.0603
Model in flash ~399 KB of 8192 KB  (4.87%)
SRAM at runtime: model is const/flash-resident; only a 40-float input buffer in RAM


So the full model is ~a few hundred KB of an 8 MB flash — a rounding error — and
needs essentially no RAM. It fits the MCU with enormous headroom. Now compile it to C
and prove the C reproduces the Python model exactly.

In [3]:
# Regenerate the float C model + validation vectors from the saved booster.
print(subprocess.run([sys.executable, 'hw/generate_c.py'], capture_output=True, text=True).stdout)

Loading data/Dataset.csv ...
Parsed 346 trees
Flattened to 20414 nodes
Base margin = 0.00000000 | max residual vs xgboost margin = 0.00e+00
Wrote hw/flagship_golden_vectors.csv (2000 rows)
Wrote hw/mcu/test_vectors.h (64 embedded vectors)
Done.



In [4]:
# Compiler-free bit-check: parse the EMITTED C node table (with its %.9g rounding)
# and run the exact C traversal in float32 against the golden vectors.
C = open('hw/mcu/sepsis_flagship_model.c').read()
def _f(t):
    t = t.strip().rstrip('f'); return np.float32('nan') if t=='NAN' else np.float32(float(t))
base = _f(re.search(r'#define SEPSIS_BASE_MARGIN \(([^)]+)\)', C).group(1))
roots = [int(x) for x in re.findall(r'\d+', re.search(r'SEPSIS_TREE_ROOTS\[[^\]]*\]\s*=\s*\{(.*?)\};', C, re.S).group(1))]
nodes = []
for m in re.finditer(r'\{([^}]*)\}', re.search(r'SEPSIS_NODES\[[^\]]*\]\s*=\s*\{(.*)\};', C, re.S).group(1)):
    p = m.group(1).split(','); nodes.append((int(p[0]), _f(p[1]), int(p[2]), int(p[3]), int(p[4]), _f(p[5])))
def c_margin(x):
    s = np.float32(base)
    for r in roots:
        n = r
        while nodes[n][0] != 255:
            ft, th, y, no, mi, _ = nodes[n]; v = x[ft]
            n = mi if np.isnan(v) else (y if np.float32(v) < th else no)
        s = np.float32(s + nodes[n][5])
    return float(s)
gv = pd.read_csv('hw/flagship_golden_vectors.csv')
FE = [c for c in gv.columns if not c.startswith('xgb_')]
Xg = gv[FE].values.astype(np.float32)
cm = np.array([c_margin(x) for x in Xg])
md_ = np.abs(cm - gv['xgb_margin'].values)
mm = int(((cm>=0) != (gv['xgb_margin'].values>=0)).sum())
print(f'vectors {len(Xg)} | max|margin diff| {md_.max():.2e} | sign mismatches {mm}')
print('PASS: generated C reproduces the flagship model' if md_.max()<1e-3 and mm==0 else 'FAIL')

vectors 2000 | max|margin diff| 5.93e-08 | sign mismatches 0
PASS: generated C reproduces the flagship model


## Why this model would be impractical as combinational logic

Counting split-nodes = the number of physical comparators a combinational
implementation needs (each a fixed-point magnitude compare on a distinct feature bus).

In [5]:
import json
def comparators(path):
    m = json.load(open(path))['learner']['gradient_booster']['model']['trees']
    return sum(sum(1 for l in t['left_children'] if l != -1) for t in m), len(m)
ce, te_ = comparators('models/sepsis_edge.json')
cs, ts_ = comparators('models/sepsis_booster.json')
print(f'EDGE (FPGA):     {te_:>3} trees, {ce:>6,} comparators, 4 feature buses')
print(f'FLAGSHIP (MCU):  {ts_:>3} trees, {cs:>6,} comparators, 40 feature buses')
print(f'--> ~{cs/ce:.0f}x the comparators over 10x the input buses if built as gates,')
print(f'    but on the M4 it is the SAME one FPU reused across cycles -> fits in flash.')

EDGE (FPGA):      40 trees,    588 comparators, 4 feature buses
FLAGSHIP (MCU):  366 trees, 10,581 comparators, 40 feature buses
--> ~18x the comparators over 10x the input buses if built as gates,
    but on the M4 it is the SAME one FPU reused across cycles -> fits in flash.


## Deploy and measure on the physical board

Build files are in **`hw/mcu/`** (see `hw/mcu/README.md`). The Simplicity Studio 6
project adds `sepsis_flagship_model.c/.h`, `test_vectors.h`, and
`app_sepsis_benchmark.c`, which times each inference with the Cortex-M4 **DWT cycle
counter** and streams results over the VCOM debug UART.

### On-device results (measured)

Captured from the BRD2605A over VCOM at 115200 8N1 — full log in
[`figures/mcu_serial_log.txt`](figures/mcu_serial_log.txt):

```
=== Sepsis flagship model on BRD2605A (SiWG917, Cortex-M4F) ===
model: 346 trees, 40 features, 20414 nodes | core 180 MHz
...
inferences: 64 | on-device vs host-reference mismatches: 0
cycles  min=2658835  avg=2810106  max=2881864
latency min=14771.30 us  avg=15611.70 us  max=16010.35 us
throughput ~64 inferences/sec
```

| Measured | Value |
|---|---|
| On-device vs host-model agreement | **64/64 vectors, 0 mismatches** (margins match to all 5 printed decimals) |
| Inference latency (avg) | **15.6 ms** (min 14.8 / max 16.0 ms) |
| Throughput | ~64 inferences/sec |
| Core clock (verified at runtime) | 180 MHz |

**Where does 15.6 ms come from?** ~2.8 M cycles per inference is ~140 cycles per node
visit — far more than the handful of compare/branch instructions need. The 400 KB node
table lives in QSPI flash (XIP), and the effectively random node-to-node access pattern
defeats the cache, so the M4 stalls on flash reads. Compute is not the bottleneck;
**memory locality is**. At the hourly cadence of vital-sign scoring this is a
~230,000x real-time margin, so no optimization is needed — but it is a nice
illustration that on MCUs the memory system, not the ALU, usually dominates.

*Board photo: `figures/mcu_board.jpg`.*

Two honest, distinct hardware claims from this project:

- **FPGA path:** 4-feature model compiled to Verilog, verified in RTL simulation — no
  physical board.
- **MCU path:** full 40-feature model compiled to C, running on **physical BRD2605A
  silicon** — 64/64 vectors bit-consistent with the host model, latency measured with
  the DWT cycle counter.

Both are generated from the same trained model, so the targets are provably the same
model — not two different implementations.


## Bedside-monitor replay on physical silicon

The benchmark above proves *correctness and speed*, but not the *clinical* behaviour.
For that, the device runs the MCU counterpart of the Vivado patient-streaming testbench
(`hw/tb_demo_patients.v`) and the Dash dashboard (`demo/app.py`) — but instead of
synthetic vitals it streams **real held-out PhysioNet-2019 test patients** hour by hour
through the flagship model, printing the **calibrated sepsis risk** and **alarm** state
each hour.

The on-device risk uses the exact same pipeline as the dashboard —
`risk = isotonic(sigmoid(margin))` — with the isotonic calibration
(`models/sepsis_isotonic.joblib`) embedded as a 104-point piecewise-linear table
(reproduces the host risk to 4.6e-8) and the same 0.037 alarm threshold. Data generated
by `hw/generate_patient_demo.py`, streamed by `hw/mcu/app_sepsis_patient_demo.c`.

Two patients replayed on the board (full log:
[`figures/mcu_patient_replay_log.txt`](figures/mcu_patient_replay_log.txt)):

```
--- Patient A: SEPTIC (held-out test patient 11575) ---
hour   HR  Temp Resp  SBP |  risk  | ALARM label
 h08   97   --  21 122 | 0.013 |   0     0
 h09  102 38.4  23 129 | 0.044 |   1     0  <-- ALARM raised
 ...
 h13  101   --  26 132 | 0.044 |   1     1  <-- sepsis warning label
 ...
 h18   92   --  22 113 | 0.139 |   1     1
device-vs-host risk mismatches: 0/22
first alarm h9 | first warning label h13 -> 4 h early warning on silicon

--- Patient B: CONTROL, non-septic (held-out test patient 99) ---
 ... risk 0.005 -> 0.029, ALARM 0 for all 22 h ...
alarm stayed quiet for all 22 h (correct: non-septic patient)

=== replay done | total device-vs-host mismatches: 0 ===
```

| On-device clinical behaviour | Result |
|---|---|
| Septic patient — early warning | **alarm at h9, sepsis label at h13 → 4 h lead time** |
| Control patient — false alarms | **0** (alarm quiet all 22 h) |
| Device-vs-host calibrated-risk agreement | **0 mismatches / 44 patient-hours** |

![Bedside-monitor replay running live on the BRD2605A](figures/mcu_live_demo.jpg)

*The BRD2605A streaming held-out ICU patients over VCOM — the risk score climbing and
the alarm firing four hours ahead of the sepsis label, running entirely on the M4.*


## Scope & honesty notes

- Inference runs on **pre-recorded test vectors** on real hardware — this is not a
  patient-connected or clinically deployed system.
- The board is on loan via the Silicon Labs workshop, not permanently owned.
- The C model is **float** (uses the M4F FPU), so it is the same model as the Python
  float booster to floating-point rounding (validated above: max margin diff ~1e-7,
  zero decision mismatches on 2000 held-out vectors).
